# SermukaiBot · Avance de proyecto 2
**Desarrollo de aplicaciones con IA (MBIA2602) · Universidad Tecmilenio**

Primera versión **multimodal** de SermukaiBot. Además de entender preguntas sobre la plataforma de inventario, ahora reconoce **fotos de artículos** (radios, uniformes, gorras…) y le dice al empleado si se registran como **Asset** o como **Stock**.

| Parte | Qué se hace | Instrucción de la práctica |
|---|---|---|
| 0 · Contexto | Recuperación del Avance 1 y arquitectura multimodal | 1 |
| 1 · Texto | Dataset de intenciones → 4 técnicas de limpieza → TF-IDF → SVM → métricas | 2, 3, 4 y 5 |
| 2 · Imagen | Fotos descargadas de la web → limpieza → ResNet50 (transfer learning) → 5 evidencias | 6, 7 y 8 |
| 3 · Integración | `responder(texto, foto)` usa los dos modelos y comunica el resultado | 9 |
| 4 · Análisis | Guía para interpretar resultados, limitaciones y siguientes pasos | 10 |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*. Luego ejecuta las celdas en orden (**Shift + Enter**).

---
# PARTE 0 · Contexto (Avance 1)

**Problema.** Una empresa que presta servicios de guardias de seguridad pasó su inventario (uniformes, radios, gorras, accesorios) de archivos de Excel a una plataforma nueva. Los empleados todavía no la dominan y la documentación formal tardará en estar lista.

**Objetivo.** Un chatbot que explique cómo operar la plataforma para acortar la curva de aprendizaje y reducir las interrupciones a supervisores por dudas básicas.

**Usuarios.** Quienes operan la plataforma (supervisores, asistentes y gerentes), con tres roles: **Lector**, **Operador** y **Administrador**.

**Alcance.** Asistencia informativa 24/7: el bot explica los pasos y el usuario ejecuta la acción. Si no entiende la consulta, responde con un mensaje de respaldo.

**Chatbot del Avance 1.** Python + NLTK con palabras clave. Flujo: saludo → identificación del tema (Assets, Stocks, Catálogo, Ubicaciones…) → intención → rol del usuario → respuesta paso a paso → confirmación → cierre.

**Concepto clave del sistema.** Cada producto del Catálogo tiene un *tipo de seguimiento*:
- **Serialized** → se rastrea por número de serie en *Inventario → Assets* (p. ej. radios).
- **Quantity** → se controla por cantidad en *Inventario → Stocks* (p. ej. uniformes, gorras, accesorios).

## ¿Qué cambia en el Avance 2?
1. Las intenciones ya no se detectan con reglas: se **aprenden** con TF-IDF + SVM.
2. Se agrega un **componente visual**: el empleado manda la foto de un artículo y el bot le dice qué es, si es *Serialized* o *Quantity* y en qué sección se registra.

```
Usuario (texto + foto opcional)
   │
   ├── texto ─► preprocesar ─► TF-IDF ─► SVM ─────────────────► intención
   │
   └── foto ──► ResNet50 congelada ─► capa softmax ───────────► artículo + confianza
                                                                     │
        responder(): intención + artículo ─► tipo (Serialized / Quantity) + sección + pasos
```

In [ ]:
# Paso 0 · Bibliotecas
!pip -q install icrawler ddgs

import os, re, glob, shutil, hashlib, logging, unicodedata
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from icrawler.builtin import BingImageCrawler
tf.keras.utils.set_random_seed(42)   # para que los resultados se repitan
print("GPU:", tf.config.list_physical_devices("GPU") or "no hay GPU (funciona, pero más lento)")

---
# PARTE 1 · Texto

## Paso 1 · Dataset de intenciones
Una **intención** es lo que el empleado quiere hacer en la plataforma. Las intenciones salen del diseño conversacional y de las conversaciones de prueba del Avance 1:

| Grupo | Intenciones |
|---|---|
| Conversación | `saludo`, `despedida`, `agradecimiento` |
| Assets (Serialized) | `crear_asset`, `asignar_asset`, `devolver_asset`, `mantenimiento` |
| Otras secciones | `stock`, `catalogo`, `ubicaciones`, `exportar`, `auditoria`, `roles` |
| Multimodal (nueva) | `clasificar_imagen` → conecta con la parte de imágenes |

14 intenciones × 15 frases = **210 frases**. Algunas se parecen a propósito (*"¿quién asignó este radio?"* es de auditoría, no de asignar) para que la evaluación sea realista.

In [ ]:
INTENCIONES = {
 'saludo': [
  'hola', 'hola, buenos días', 'buenas tardes', 'qué tal', 'hola SermukaiBot', 'buenas noches',
  'hey, hola', 'buen día', 'saludos', 'hola, ¿cómo estás?', 'qué onda', 'hola, necesito ayuda con el sistema',
  'buenas', 'hola, ¿me ayudas?', 'muy buenos días'],
 'despedida': [
  'adiós', 'hasta luego', 'nos vemos', 'bye', 'me voy, eso fue todo', 'hasta mañana',
  'eso es todo, adiós', 'chao', 'salir', 'hasta pronto', 'ya me tengo que ir',
  'me despido', 'terminar la conversación', 'hasta la próxima', 'ya terminé, bye'],
 'agradecimiento': [
  'gracias', 'muchas gracias', 'te lo agradezco', 'gracias por la ayuda', 'excelente, gracias',
  'mil gracias', 'muy amable', 'perfecto, gracias', 'gracias, ya quedó', 'sí, me sirvió, gracias',
  'ok, gracias', 'te agradezco mucho', 'súper, gracias', 'gracias SermukaiBot', 'muy útil, gracias'],
 'crear_asset': [
  '¿cómo creo un nuevo asset?', '¿cómo doy de alta un radio en el sistema?',
  'ya tengo objetos en catálogo, ¿cómo los paso a un asset?', 'quiero registrar un activo nuevo',
  '¿cómo agrego un radio con su número de serie?', '¿dónde está el botón de nuevo activo?',
  '¿cómo registro un equipo serializado?', 'no me aparece el producto al crear el asset',
  '¿qué datos pide el formulario de nuevo activo?', '¿cómo capturo el número de serie de un radio?',
  'dar de alta un activo', '¿puedo subir una foto al crear un asset?', 'registrar radios nuevos en assets',
  '¿cómo añado un activo al inventario?', 'alta de un nuevo equipo con número de serie'],
 'asignar_asset': [
  '¿cómo le asigno una radio a un guardia?', 'ya tengo el asset creado, ¿cómo lo entrego a un guardia?',
  'quiero asignar un activo a un empleado', '¿cómo entrego un radio a un elemento?',
  'asignar equipo a una persona', '¿cómo pongo un asset a nombre de un guardia?',
  '¿dónde elijo la ubicación de servicio al asignar?', '¿cómo registro que un guardia se llevó un radio?',
  '¿puedo poner fecha de devolución esperada al asignar?', 'entregar un activo a un guardia del servicio',
  '¿por qué no me deja asignar el radio?', 'resguardo de radio para un guardia',
  '¿cómo cambio el estado de un activo a asignado?', 'asignación de equipo a personal',
  'quiero darle un radio a un nuevo guardia'],
 'devolver_asset': [
  '¿cómo devuelvo un asset?', 'el guardia regresó el radio, ¿cómo lo registro?',
  '¿cómo regreso un activo asignado a stock?', 'registrar la devolución de un equipo',
  'un elemento entregó su radio, ¿qué hago?', '¿dónde está el botón de devolver?',
  'quiero quitarle el radio asignado a un guardia', '¿cómo marco que un activo ya fue devuelto?',
  'devolución de un activo', 'el guardia renunció y entregó su equipo', '¿qué fecha pongo en la devolución?',
  '¿cómo libero un asset asignado?', 'regresar un radio al almacén', 'recibir de vuelta un activo',
  '¿y cómo lo devuelvo?'],
 'mantenimiento': [
  '¿cómo mando un equipo a mantenimiento?', 'un radio dejó de funcionar, ¿cómo lo marco en el sistema?',
  'ya quedó reparado, ¿cómo lo regreso a stock?', '¿cómo reprogramo un mantenimiento?',
  '¿cómo cancelo un mantenimiento?', 'enviar un radio a reparación', 'el radio está descompuesto',
  '¿dónde veo los equipos en reparación?', 'completar un mantenimiento pendiente',
  'registrar una falla de un equipo', 'nuevo mantenimiento',
  '¿cómo cambio la fecha programada del mantenimiento?', 'la batería del radio ya no carga',
  '¿qué pasa con el activo cuando termina la reparación?', 'reportar un equipo dañado'],
 'stock': [
  '¿cómo registro la entrega de uniformes?', '¿qué significa stock bajo?', '¿cómo actualizo la cantidad de gorras?',
  'llegaron camisas nuevas, ¿cómo las registro?', '¿cómo doy entrada a uniformes en el almacén?',
  'entregué cinco gorras, ¿cómo lo descuento?', '¿dónde veo cuántos chalecos quedan?', 'alerta de stock mínimo',
  '¿cómo registro una entrada de botas?', 'editar la cantidad de un ítem de stock',
  '¿por qué aparece stock bajo en el dashboard?', 'control de uniformes por cantidad',
  '¿cómo agrego stock en una ubicación?', 'descontar accesorios entregados', 'quedan pocas linternas, ¿dónde lo veo?'],
 'catalogo': [
  '¿qué diferencia hay entre asset y stock?', '¿cómo creo un producto en catálogo?',
  '¿qué es el tipo de seguimiento serialized?', '¿qué significa quantity en el catálogo?',
  'quiero agregar un nuevo producto', '¿para qué sirve la sección de catálogo?',
  '¿puedo importar productos desde un csv?', '¿cómo cambio la categoría de un producto?',
  '¿cómo edito un producto del catálogo?', '¿un radio va como serializado o por cantidad?',
  'nuevo producto en catálogo', '¿cómo defino la alerta mínima de un producto?', '¿se puede duplicar un producto?',
  '¿cómo archivo un producto que ya no usamos?', 'tipos de producto del inventario'],
 'ubicaciones': [
  'ubicaciones', '¿cómo creo una nueva ubicación?', '¿cómo elimino una ubicación?', '¿cómo desactivo una ubicación?',
  'quiero agregar un nuevo sitio de servicio', '¿dónde registro un nuevo almacén?', '¿para qué sirven las ubicaciones?',
  'dar de alta la dirección de un cliente', '¿cómo edito la dirección de una ubicación?', 'nueva ubicación',
  '¿por qué no aparece mi ubicación al asignar?', 'borrar una ubicación que ya no se usa',
  'agregar una sucursal al sistema', 'lugares donde se guarda el inventario', '¿cómo cambio el nombre de una ubicación?'],
 'exportar': [
  '¿cómo exporto la tabla a excel?', 'quiero descargar el inventario', 'exportar assets a csv',
  '¿puedo exportar solo algunas filas?', 'descargar la lista de stocks en excel', '¿cómo saco un reporte del catálogo?',
  'exportar selección', '¿en qué formato puedo descargar la tabla?', 'bajar la información a un archivo',
  'necesito el inventario en excel para mi jefe', '¿dónde está el botón de exportar?', 'generar un archivo xlsx',
  '¿se puede exportar a csv?', 'exportar los activos asignados', 'descargar reporte de inventario'],
 'auditoria': [
  '¿quién modificó este registro?', '¿dónde veo el historial de cambios?', '¿cómo sé quién borró un producto?',
  'auditoría', 'quiero ver los cambios de ayer', '¿quién asignó este radio?', 'bitácora de movimientos del sistema',
  '¿cómo filtro la auditoría por fechas?', 'ver valores anteriores de un registro',
  '¿quién editó la cantidad de uniformes?', 'rastrear cambios de un usuario', 'historial de acciones',
  '¿dónde consulto los movimientos de un activo?', 'registro de cambios por entidad', '¿qué usuario hizo este cambio?'],
 'roles': [
  '¿qué puedo hacer con mi rol?', 'soy operador', 'soy lector', 'soy administrador', '¿qué permisos tiene un lector?',
  '¿por qué no me aparecen las opciones de editar?', '¿cómo pido más permisos?',
  '¿qué diferencia hay entre operador y administrador?', 'no tengo permiso para crear registros',
  '¿quién puede eliminar ubicaciones?', '¿cómo cambio el rol de un usuario?', 'mi rol es operador',
  'tipos de usuario del sistema', '¿un lector puede exportar?', 'permisos de administrador'],
 'clasificar_imagen': [
  'te mando una foto, ¿qué artículo es?', '¿puedes identificar este equipo en la imagen?',
  'quiero subir una foto para saber cómo registrarlo', 'analiza esta imagen por favor',
  '¿qué artículo aparece en la foto?', 'mira esta foto, ¿es asset o stock?', 'clasifica esta imagen',
  'tengo una foto de un equipo, ¿qué es?', '¿me dices qué producto es el de la imagen?', 'adjunto foto del artículo',
  'reconoce el objeto de esta fotografía', 'te comparto una imagen del inventario', '¿qué ves en esta foto?',
  'esta es la foto, ¿en qué sección lo registro?', 'identifica el artículo de la imagen'],
}

frases      = [f for k, v in INTENCIONES.items() for f in v]
intenciones = [k for k, v in INTENCIONES.items() for f in v]
print(len(frases), "frases ·", len(INTENCIONES), "intenciones ·", len(frases) // len(INTENCIONES), "frases por intención")

## Paso 2 · Preprocesamiento: las 4 técnicas
1. **Tokenización:** cortar la frase en palabras.
2. **Normalización:** minúsculas y sin acentos (*Catálogo → catalogo*).
3. **Quitar stopwords:** borrar palabras vacías (*el, de, las, soy…*). Conservamos *como, donde, cuando, que, quien* y *no*, porque distinguen intenciones (*"¿quién modificó…?"* es auditoría; *"no me aparece…"* es un problema).
4. **Stemming:** dejar la raíz de cada palabra (*asigno, asignar, asignación → asign*).

In [ ]:
stemmer = SnowballStemmer("spanish")
def sin_acentos(t):
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")
CONSERVAR = {"como", "donde", "cuando", "que", "quien", "no"}
STOPWORDS = {sin_acentos(w) for w in stopwords.words("spanish")} - CONSERVAR

def preprocesar(texto, mostrar=False):
    tokens  = re.findall(r"[a-záéíóúüñ]+", texto.lower())   # 1 tokenizar
    normal  = [sin_acentos(t) for t in tokens]                # 2 normalizar
    sin_sw  = [t for t in normal if t not in STOPWORDS]       # 3 quitar stopwords
    raices  = [stemmer.stem(t) for t in sin_sw]               # 4 stemming
    if mostrar:
        print("Frase:          ", texto)
        for paso, valor in [("1 Tokens", tokens), ("2 Normalizado", normal),
                            ("3 Sin stopwords", sin_sw), ("4 Stemming", raices)]:
            print(f"{paso:<16}", valor)
        print()
    return " ".join(raices)

preprocesar("¿Cómo le asigno una radio a un guardia?", mostrar=True)
preprocesar("¿Quién modificó este registro en Auditoría?", mostrar=True);

## Paso 3 · TF-IDF + SVM: entrenar y evaluar
- **TF-IDF** convierte cada frase en números; da más peso a las palabras que distinguen una intención de otra (*asign*, *devuelv*, *export*).
- **SVM lineal** es el clasificador que aprende a pasar de esos números a la intención.
- Primero comparamos **SVM vs. Naive Bayes** con *validación cruzada* (5 particiones distintas). Es más confiable que un solo examen, porque con 210 frases un solo examen depende mucho de qué frases le tocaron.
- Luego separamos **75 % para entrenar** y **25 % para el examen** (frases que el modelo nunca vio), con la misma proporción de cada intención (`stratify`).

En el reporte: **precision** = cuando dijo X, ¿acertó?; **recall** = de todos los X reales, ¿cuántos encontró?; **f1** = balance de ambas.

In [ ]:
X = [preprocesar(f) for f in frases]

from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for nombre, clf in [("SVM lineal", LinearSVC()), ("Naive Bayes", MultinomialNB(alpha=0.1))]:
    s = cross_val_score(make_pipeline(TfidfVectorizer(), clf), X, intenciones, cv=cv)
    print(f"Validación cruzada · {nombre:<12} accuracy {s.mean():.3f} ± {s.std():.3f}")
print("→ Nos quedamos con SVM\n")

X_tr, X_te, y_tr, y_te = train_test_split(X, intenciones, test_size=0.25, stratify=intenciones, random_state=42)

modelo_texto = make_pipeline(TfidfVectorizer(), LinearSVC())
modelo_texto.fit(X_tr, y_tr)
pred = modelo_texto.predict(X_te)

p, r, f1, _ = precision_recall_fscore_support(y_te, pred, average="macro", zero_division=0)
print(f"Frases de examen: {len(X_te)}")
print(f"Accuracy: {accuracy_score(y_te, pred):.3f} · Precision: {p:.3f} · Recall: {r:.3f} · F1: {f1:.3f}  (promedio macro)\n")
print(classification_report(y_te, pred, zero_division=0))

ETIQUETAS_TXT = list(INTENCIONES)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_te, pred, labels=ETIQUETAS_TXT), annot=True, fmt="d", cmap="Blues",
            xticklabels=ETIQUETAS_TXT, yticklabels=ETIQUETAS_TXT, cbar=False)
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("Texto · Matriz de confusión de intenciones")
plt.tight_layout(); plt.show()

## Paso 4 · Analizar por intención
La práctica pide **relacionar los resultados con las intenciones**. Revisa qué frases fallaron y explica por qué en tu PDF. Por ejemplo, una frase de auditoría que contiene *"asignó"* puede irse a `asignar_asset` porque comparte la raíz *asign*.

In [ ]:
originales = dict(zip(X, frases))
errores_txt = [(originales[x], real, p_) for x, real, p_ in zip(X_te, y_te, pred) if real != p_]
print(f"{len(errores_txt)} errores de {len(X_te)} frases de examen\n")
for frase, real, p_ in errores_txt:
    print(f"«{frase}»  real: {real}  →  predijo: {p_}")

# Reentrenamos con todas las frases para usar el modelo en el asistente
modelo_texto.fit(X, intenciones)

---
# PARTE 2 · Imagen

## Paso 5 · Selección de clases
Las clases son los artículos que la empresa inventaría (Avance 1: uniformes, radios, gorras y accesorios). Dividimos "accesorios" en objetos concretos para poder buscarlos. Cada clase se relaciona con su **tipo de seguimiento** en el Catálogo:

| Clase | Tipo de seguimiento | Sección donde se gestiona |
|---|---|---|
| radio | Serialized | Inventario → Assets |
| uniforme | Quantity | Inventario → Stocks |
| gorra | Quantity | Inventario → Stocks |
| chaleco | Quantity | Inventario → Stocks |
| botas | Quantity | Inventario → Stocks |
| linterna | Quantity | Inventario → Stocks |

Las fotos se **descargan de la web** con 4 búsquedas por clase (en español y en inglés), para tener variedad de marcas, fondos y ángulos.

✏️ Si en tu inventario real algún artículo se lleva distinto (por ejemplo, linternas con número de serie), cambia su fila en `TIPO`.

In [ ]:
CLASES   = ["radio", "uniforme", "gorra", "chaleco", "botas", "linterna"]
NOMBRE   = ["un radio", "un uniforme", "una gorra", "un chaleco", "unas botas", "una linterna"]
TIPO     = ["Serialized", "Quantity", "Quantity", "Quantity", "Quantity", "Quantity"]
SECCION  = {"Serialized": "Inventario → Assets", "Quantity": "Inventario → Stocks"}

BUSQUEDAS = {
    "radio":    ["radio walkie talkie Motorola", "radio portátil de dos vías",
                 "handheld two way radio", "walkie talkie guardia de seguridad"],
    "uniforme": ["uniforme de guardia de seguridad", "camisa de uniforme de seguridad privada",
                 "security guard uniform", "uniforme de vigilante"],
    "gorra":    ["gorra de seguridad privada", "gorra táctica negra",
                 "security guard cap", "gorra beisbolera azul marino"],
    "chaleco":  ["chaleco táctico", "chaleco reflejante de seguridad",
                 "security guard vest", "chaleco para guardia de seguridad"],
    "botas":    ["botas tácticas negras", "botas de seguridad industrial",
                 "tactical boots", "botas para guardia de seguridad"],
    "linterna": ["linterna táctica LED", "lámpara de mano recargable",
                 "tactical flashlight", "linterna para guardia de seguridad"],
}

## Paso 6 · Descargar las fotos
Se piden **40 fotos por búsqueda** (≈160 por clase) a Bing Imágenes. Si una clase junta menos de 60, se intenta con DuckDuckGo como respaldo. Tarda entre 3 y 8 minutos.

💾 **Importante:** la web cambia, así que cada descarga trae fotos distintas. Al final del Paso 7 se guarda todo en `sermukai_imagenes.zip`. **Descárgalo** del panel *Archivos*. La próxima vez, súbelo antes de ejecutar esta celda: el notebook lo usará en lugar de descargar, y obtendrás los mismos resultados que pusiste en tu PDF.

In [ ]:
ZIP = "sermukai_imagenes.zip"
POR_BUSQUEDA = 40

def descargar_bing(clase):
    for q in BUSQUEDAS[clase]:
        BingImageCrawler(downloader_threads=4, storage={"root_dir": f"crudo/{clase}"},
                         log_level=logging.CRITICAL).crawl(keyword=q, max_num=POR_BUSQUEDA, file_idx_offset="auto")

def descargar_ddg(clase):                      # respaldo por si Bing devuelve pocas fotos
    import requests
    from ddgs import DDGS
    for j, q in enumerate(BUSQUEDAS[clase]):
        for k, r in enumerate(DDGS().images(q, max_results=POR_BUSQUEDA)):
            try:
                with open(f"crudo/{clase}/ddg_{j}_{k}", "wb") as f:
                    f.write(requests.get(r["image"], timeout=10).content)
            except Exception:
                pass                           # enlace roto: se ignora

if os.path.exists(ZIP):
    shutil.unpack_archive(ZIP, ".")
    print("Usando las fotos guardadas en", ZIP)
else:
    for c in CLASES:
        os.makedirs(f"crudo/{c}", exist_ok=True)
        descargar_bing(c)
        if len(os.listdir(f"crudo/{c}")) < 60:
            try:
                descargar_ddg(c)
            except Exception as e:
                print("   respaldo DuckDuckGo no disponible:", e)
        print(f"{c:<9} {len(os.listdir(f'crudo/{c}')):>4} archivos descargados")

## Paso 7 · Limpiar y revisar las fotos
Las fotos de internet traen basura, así que se limpian automáticamente:
- Se descartan archivos **dañados** o que no son imagen.
- Se descartan imágenes **muy pequeñas** (menos de 120 px).
- Se descartan **repetidas**, también entre clases (la misma foto no puede estar en "gorra" y en "uniforme").
- Todas se convierten a **JPG RGB de 224 × 224**, el tamaño que espera ResNet50.

In [ ]:
def limpiar():
    vistos, conteo = set(), {}
    for c in CLASES:
        os.makedirs(f"datos/{c}", exist_ok=True)
        n, descartadas = 0, 0
        for ruta in sorted(glob.glob(f"crudo/{c}/*")):
            try:
                img = Image.open(ruta).convert("RGB")
            except Exception:
                descartadas += 1; continue                       # dañada o no es imagen
            if min(img.size) < 120:
                descartadas += 1; continue                       # demasiado pequeña
            huella = hashlib.md5(img.resize((16, 16)).convert("L").tobytes()).hexdigest()
            if huella in vistos:
                descartadas += 1; continue                       # repetida
            vistos.add(huella)
            img.resize((224, 224)).save(f"datos/{c}/{c}_{n:03d}.jpg", quality=90)
            n += 1
        conteo[c] = (n, descartadas)
    return conteo

if not os.path.isdir("datos"):
    for c, (n, d) in limpiar().items():
        print(f"{c:<9} {n:>4} útiles · {d:>3} descartadas")

### Revisión manual (recomendada)
La limpieza automática no detecta fotos que **no corresponden a la clase** (un dibujo, un collage, un modelo con uniforme y gorra a la vez…). Revisa la muestra de abajo. Si ves fotos malas, bórralas en el panel *Archivos → datos/<clase>/* y vuelve a ejecutar esta celda para ver otra muestra.

📝 Anota cuántas borraste por clase: es un dato concreto para la sección de limitaciones de tu PDF.

In [ ]:
rng = np.random.default_rng()
fig, ejes = plt.subplots(len(CLASES), 8, figsize=(16, 2.3 * len(CLASES)))
for i, c in enumerate(CLASES):
    fotos = sorted(glob.glob(f"datos/{c}/*.jpg"))
    muestra = rng.choice(fotos, size=min(8, len(fotos)), replace=False)
    for j, eje in enumerate(ejes[i]):
        eje.axis("off")
        if j < len(muestra):
            eje.imshow(Image.open(muestra[j]))
            eje.set_title(os.path.basename(muestra[j]), fontsize=7)
plt.suptitle("Muestra del conjunto de imágenes (una fila por clase)"); plt.tight_layout(); plt.show()

In [ ]:
# Cuando termines de revisar, guarda el conjunto final (y descárgalo del panel Archivos)
shutil.make_archive("sermukai_imagenes", "zip", ".", "datos")
print("Guardado:", ZIP, f"({os.path.getsize(ZIP) / 1e6:.1f} MB)")
# from google.colab import files; files.download(ZIP)   # ← descomenta para descargarlo directo

## Paso 8 · Separar entrenamiento y examen
Separamos **80 % para entrenar** y **20 % para el examen**, **antes** de entrenar y con la misma proporción por clase. Las fotos del examen solo se usan para calcular las métricas.

In [ ]:
archivos, etiquetas = [], []
for i, c in enumerate(CLASES):
    fotos = sorted(glob.glob(f"datos/{c}/*.jpg"))
    archivos += fotos
    etiquetas += [i] * len(fotos)
archivos, etiquetas = np.array(archivos), np.array(etiquetas)

tr, te = train_test_split(np.arange(len(archivos)), test_size=0.20, stratify=etiquetas, random_state=42)
print(len(archivos), "fotos ·", len(tr), "para entrenar ·", len(te), "para el examen\n")
for i, c in enumerate(CLASES):
    print(f"{c:<9} total {np.sum(etiquetas == i):>4} · entrenar {np.sum(etiquetas[tr] == i):>4} · examen {np.sum(etiquetas[te] == i):>3}")

## Paso 9 · ResNet50 con transfer learning
**ResNet50** ya fue entrenada con más de un millón de fotos (ImageNet): ya sabe "ver" bordes, texturas y formas.
- La **congelamos** (no se modifica) y la usamos para convertir cada foto en **2,048 números** que la describen.
- Encima entrenamos una **capa nueva** de 6 salidas (una por artículo) con **softmax**, que da probabilidades.
- `class_weight` compensa si alguna clase quedó con menos fotos después de la limpieza.

Con GPU tarda 1 o 2 minutos.

In [ ]:
base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3), pooling="avg")
base.trainable = False                                   # congelada

def cargar(ruta):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    return tf.image.resize(img, (224, 224))

def describir(rutas):                                    # foto → 2048 números
    salida = []
    for i in range(0, len(rutas), 32):
        lote = tf.stack([cargar(r) for r in rutas[i:i + 32]])
        salida.append(base(preprocess_input(lote), training=False).numpy())
    return np.concatenate(salida)

F = describir(archivos)

conteo_tr = np.bincount(etiquetas[tr], minlength=len(CLASES))
pesos = {i: len(tr) / (len(CLASES) * n) for i, n in enumerate(conteo_tr)}

cabeza = models.Sequential([layers.Input((2048,)), layers.Dropout(0.4),
                            layers.Dense(len(CLASES), activation="softmax")])
cabeza.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist = cabeza.fit(F[tr], etiquetas[tr], epochs=25, batch_size=32, validation_split=0.1,
                  class_weight=pesos, verbose=0)

plt.plot(hist.history["accuracy"], label="entrenamiento")
plt.plot(hist.history["val_accuracy"], label="validación")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.title("Entrenamiento de la capa nueva"); plt.legend(); plt.show()
print("Modelo entrenado ✅")

## Paso 10 · Las 5 evidencias
La práctica pide **las 5**: accuracy, precision, recall, F1-score y matriz de confusión. Se calculan con las fotos del **examen**.

En la matriz, las filas son la clase real y las columnas lo que dijo el modelo. La **diagonal** son los aciertos; lo demás son errores. Debajo se muestran las fotos que falló, para que expliques **por qué** se equivocó.

In [ ]:
pred_img = cabeza.predict(F[te], verbose=0).argmax(axis=1)
real_img = etiquetas[te]

p, r, f1, _ = precision_recall_fscore_support(real_img, pred_img, average="macro", zero_division=0)
print(f"1 Accuracy:  {accuracy_score(real_img, pred_img):.3f}")
print(f"2 Precision: {p:.3f}")
print(f"3 Recall:    {r:.3f}")
print(f"4 F1-score:  {f1:.3f}   (precision, recall y F1 son promedio macro de las 6 clases)\n")
print(classification_report(real_img, pred_img, target_names=CLASES, digits=3, zero_division=0))

sns.heatmap(confusion_matrix(real_img, pred_img), annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASES, yticklabels=CLASES)
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("5 · Matriz de confusión"); plt.show()

errores_img = np.where(pred_img != real_img)[0]
print(f"{len(errores_img)} errores de {len(te)} fotos de examen")
n = min(10, len(errores_img))
if n:
    fig, ejes = plt.subplots(1, n, figsize=(2 * n, 2.8), squeeze=False)
    for eje, k in zip(ejes[0], errores_img[:n]):
        eje.imshow(Image.open(archivos[te[k]])); eje.axis("off")
        eje.set_title(f"real: {CLASES[real_img[k]]}\npredijo: {CLASES[pred_img[k]]}", fontsize=8)
    plt.tight_layout(); plt.show()

---
# PARTE 3 · Integración texto + imagen

## Paso 11 · La función `responder()`
Une los dos modelos en **3 operaciones**:
1. **Procesa el texto** → obtiene la intención. Si ninguna palabra de la frase es conocida, no adivina: responde con el mensaje de respaldo del Avance 1.
2. Si hay foto → **clasifica la imagen** con ResNet50 y obtiene el artículo y su confianza.
3. **Comunica el resultado** combinando ambos:
   - Confianza menor a 60 % → pide otra foto en lugar de adivinar.
   - El artículo define el **tipo de seguimiento** (Serialized / Quantity) y la **sección**.
   - Si el usuario pide una acción de Assets (asignar, devolver, mantenimiento) sobre un artículo **Quantity**, el bot le avisa que esos artículos no se asignan, se controlan por cantidad. Este es un error común entre los empleados nuevos.

✏️ Las respuestas son versiones cortas de las del Avance 1; ajústalas a tu plataforma.

In [ ]:
RESPUESTAS = {
    "saludo": "Bienvenido al bot de la plataforma de inventario. ¿En qué te puedo ayudar? Temas: Assets, Stocks, "
              "Catálogo, Mantenimientos, Ubicaciones, Exportar, Auditoría y Roles. También puedes mandarme la foto "
              "de un artículo y te digo cómo registrarlo.",
    "despedida": "¡Hasta luego! Gracias por consultar SermukaiBot.",
    "agradecimiento": "¡De nada! ¿Necesitas orientación sobre otro tema?",
    "crear_asset": "Primero debe existir en Catálogo un producto tipo «Serialized». Abre Inventario → Assets, pulsa "
                   "«Nuevo activo», elige el «Tipo de ítem», captura el «Número de serie» y pulsa «Añadir». "
                   "(Requiere rol Operador o superior.)",
    "asignar_asset": "El activo debe estar «En stock». Abre Inventario → Assets, entra a su detalle y pulsa «Asignar». "
                     "Escribe el «Nombre del empleado», elige la «Ubicación de servicio» y confirma con «Asignar». "
                     "(Requiere rol Operador o superior.)",
    "devolver_asset": "Abre Inventario → Assets, entra al detalle del activo «Asignado» y pulsa «Devolver». Indica la "
                      "«Fecha de devolución» y confirma. El activo regresa a «En stock».",
    "mantenimiento": "En Inventario → Assets abre el activo y pulsa «Enviar a mantenimiento». Describe la falla, elige "
                     "la «Fecha programada» y confirma. Para completarlo, reprogramarlo o cancelarlo usa "
                     "Operación → Mantenimientos.",
    "stock": "Los Stocks se controlan por cantidad. Abre Inventario → Stocks, menú «⋯» del ítem → «Editar», actualiza "
             "la «Cantidad» y pulsa «Guardar». «Stock bajo» aparece cuando la cantidad queda debajo del mínimo.",
    "catalogo": "El Catálogo define los productos. «Serialized» se rastrea por número de serie en Assets (radios); "
                "«Quantity» se maneja por cantidad en Stocks (uniformes, gorras, accesorios).",
    "ubicaciones": "Abre Operación → Ubicaciones y pulsa «Nueva ubicación»: escribe el «Nombre» y, si quieres, "
                   "«Dirección» y «Descripción». Eliminar una ubicación solo está disponible para Administradores.",
    "exportar": "En Assets, Stocks o Catálogo pulsa «Exportar» y elige «Excel (.xlsx)» o «CSV (.csv)». También "
                "puedes marcar filas y usar «Exportar selección».",
    "auditoria": "Abre Administración → Auditoría: cada cambio guarda Fecha, Acción, Entidad, Usuario y los valores "
                 "anteriores y nuevos. Filtra por fechas, entidad o acción. (Requiere rol Administrador.)",
    "roles": "Lector: consulta, busca, filtra y exporta. Operador: además crea y edita activos, stock, productos y "
             "ubicaciones, y asigna, devuelve o manda a mantenimiento. Administrador: además elimina ubicaciones y "
             "consulta Auditoría.",
}
RESPALDO = ("No identifiqué tu consulta. Menciona una sección (Assets, Stocks, Catálogo, Mantenimientos, "
            "Ubicaciones) y lo que quieres hacer, o mándame la foto de un artículo.")
PASOS = {
    "Serialized": "Para registrarlo: Inventario → Assets → «Nuevo activo», elige el «Tipo de ítem», captura el "
                  "«Número de serie» y pulsa «Añadir».",
    "Quantity": "Para registrarlo: Inventario → Stocks, menú «⋯» del ítem → «Editar», actualiza la «Cantidad» "
                "y pulsa «Guardar».",
}
ACCIONES_DE_ASSET = {"crear_asset", "asignar_asset", "devolver_asset", "mantenimiento"}
UMBRAL = 0.60

def entender_texto(texto):                                        # 1 · texto
    x = preprocesar(texto)
    if modelo_texto[0].transform([x]).nnz == 0:                   # ninguna palabra conocida
        return None
    return modelo_texto.predict([x])[0]

def clasificar_foto(ruta):                                        # 2 · imagen
    prob = cabeza.predict(describir([ruta]), verbose=0)[0]
    c = int(prob.argmax())
    return c, float(prob[c])

def responder(texto, foto=None):
    intencion = entender_texto(texto)
    if foto is None:
        if intencion is None:
            return RESPALDO
        if intencion == "clasificar_imagen":
            return "Adjunta la foto del artículo y te digo cómo registrarlo."
        return RESPUESTAS[intencion]

    c, confianza = clasificar_foto(foto)                          # 3 · comunicar
    if confianza < UMBRAL:
        return (f"No estoy seguro de qué artículo es (mi mejor opción es {CLASES[c]} con {confianza:.0%}). "
                "Toma otra foto con buena luz, de frente y con el artículo solo.")
    tipo = TIPO[c]
    mensaje = f"Parece {NOMBRE[c]} ({confianza:.0%}). En el Catálogo es tipo «{tipo}»: se gestiona en {SECCION[tipo]}. "
    if intencion in ACCIONES_DE_ASSET and tipo == "Quantity":
        return mensaje + ("Ojo: los artículos tipo Quantity no se asignan ni se mandan a mantenimiento como un Asset; "
                          "se entregan descontando la cantidad. " + PASOS["Quantity"])
    if intencion in ACCIONES_DE_ASSET:
        return mensaje + RESPUESTAS[intencion]
    return mensaje + PASOS[tipo]

## Paso 12 · Prueba de la conversación
La prueba cubre los casos que importan:
- Conversación normal, sin foto.
- Foto de un **radio** (Serialized).
- Foto de una **gorra** con una acción de Assets (el bot debe avisar que es Quantity).
- Una pregunta fuera del dominio (respaldo de texto).
- Una foto que **no es del inventario** (un perro), para probar el umbral de confianza.

📸 Toma captura de esta salida para tu PDF: es la evidencia de la integración.

In [ ]:
def foto_de(clase):                     # primera foto del examen de esa clase (el modelo no la vio al entrenar)
    i = CLASES.index(clase)
    return archivos[te[etiquetas[te] == i][0]]

perro = tf.keras.utils.get_file("perro.jpg",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg")

conversacion = [
    ("hola", None),
    ("Te mando una foto, ¿qué artículo es?", foto_de("radio")),
    ("¿Cómo le asigno esto a un guardia?", foto_de("gorra")),
    ("¿Cómo mando un radio a mantenimiento?", None),
    ("quiero clasificar una foto", None),
    ("¿Cuál es el precio del dólar hoy?", None),
    ("¿Qué artículo es este?", perro),
    ("gracias", None),
]
for texto, f in conversacion:
    if f is not None:
        plt.figure(figsize=(2.2, 2.2)); plt.imshow(Image.open(f).convert("RGB")); plt.axis("off"); plt.show()
    print("👤", texto, "📎" if f is not None else "")
    print("🤖", responder(texto, f), "\n")

In [ ]:
# ✏️ Prueba con tu propia foto: súbela al panel Archivos y cambia el nombre
# print(responder("¿Cómo registro esto?", "mi_foto.jpg"))

---
# PARTE 4 · Para tu PDF: análisis y conclusiones
Escribe con **tus** números (cambian según las fotos descargadas):

**1. Interpretación**
- *Texto (Pasos 3 y 4):* accuracy y F1 macro. ¿Qué intenciones fallaron y con cuál se confundieron? Busca raíces compartidas: *asign* (asignar vs. auditoría), *ubic* (ubicaciones vs. roles), *registr* (crear_asset vs. stock vs. devolver).
- *Imagen (Paso 10):* las 5 evidencias. ¿Qué clase tiene el recall más bajo? Revisa las fotos falladas. Confusiones esperables: **uniforme ↔ chaleco** (las fotos de personas traen ambos), **gorra** dentro de fotos de uniformes, **radio ↔ linterna** (objetos negros que se sostienen con la mano).
- *Integración (Paso 12):* ¿el bot avisó correctamente que la gorra es Quantity? ¿El umbral detuvo la foto del perro?

**2. Limitaciones** (elige las que tus números respalden)
- Las fotos vienen de la web, no del almacén de la empresa: son fotos de catálogo con fondo blanco. Con fotos reales tomadas con celular, el desempeño probablemente baje (*cambio de dominio*).
- Ruido en las etiquetas: cuántas fotos tuviste que borrar en la revisión manual.
- Softmax siempre reparte el 100 % entre las 6 clases. Si el perro salió con confianza alta, el modelo no sabe decir "esto no es del inventario".
- El dataset de texto es pequeño (15 frases por intención) y lo escribió una sola persona, no viene de empleados reales. Compara la validación cruzada con el examen del Paso 3: si difieren mucho, el resultado depende de qué frases tocaron en el examen.
- El respaldo de texto solo se activa cuando **ninguna** palabra es conocida. *"¿Qué clima hará mañana?"* se clasifica como `despedida` porque *mañana* aparece en *"hasta mañana"*.
- El rol del usuario todavía no se usa en la respuesta.

**3. Acciones para la versión final**
- Tomar fotos reales del inventario de la empresa, al menos para el examen, y comparar contra las de la web.
- Agregar una clase **"otro"** con objetos que no son del inventario, o calibrar el umbral con fotos fuera del dominio.
- Hacer *fine-tuning* de los últimos bloques de ResNet50 con *data augmentation*, y comparar contra **EfficientNetB0**.
- Ampliar las frases con preguntas reales de los empleados y volver a medir.
- Leer el rol desde el token de sesión (conclusión del Avance 1) para personalizar la respuesta, e integrar la carga de fotos en el botón "ChatBot" de la plataforma.